In [7]:
import os
import gymnasium as gym
os.environ["D3RLPY_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data"
os.makedirs(os.environ["D3RLPY_DATASETS_PATH"], exist_ok=True)
os.environ["MINARI_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data/minari_data"
os.makedirs(os.environ["MINARI_DATASETS_PATH"], exist_ok=True)

import d3rlpy

dataset, env = d3rlpy.datasets.get_cartpole()
env = gym.make("CartPole-v1",max_episode_steps=200)

seed = 1

dataset_name = "cartpole"
# fix seed
d3rlpy.seed(seed)

2025-07-26 18:55.48 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int32')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(4,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-07-26 18:55.48 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-07-26 18:55.48 [info     ] Action size has been automatically determined. action_size=2


In [8]:
from d3rlpy.logging import UnifiedFileAdapterFactory

batch_size = 128
context_size = 30

dt = d3rlpy.algos.DiscreteDecisionTransformerConfig(
    batch_size=batch_size,
    context_size=context_size,
    learning_rate=6e-4,
    activation_type="gelu",
    embed_activation_type="tanh",
    # encoder_factory=d3rlpy.models.PixelEncoderFactory(
    #     feature_size=128, exclude_last_activation=True
    # ),  # Nature DQN
    num_heads=8,
    num_layers=6,
    attn_dropout=0.1,
    embed_dropout=0.1,
    optim_factory=d3rlpy.optimizers.GPTAdamWFactory(
        betas=(0.9, 0.95),
        weight_decay=0.1,
        clip_grad_norm=1.0,
    ),
    warmup_tokens=512 * 20,
    final_tokens=2 * 500000 * context_size * 3,
    # observation_scaler=d3rlpy.preprocessing.PixelObservationScaler(),
    max_timestep=200, # TODO
    position_encoding_type=d3rlpy.PositionEncodingType.GLOBAL,
    compile_graph=False,
).create(device="cuda:0")

dt.fit(
    dataset,
    n_steps=100000,#100000,
    n_steps_per_epoch=1000,#1000,
    save_interval=100,
    eval_env=env,
    eval_target_return=200,
    experiment_name=f"Discrete_DT_Cartpole",
    n_trials=50,
    eval_gaps=1,
    eval_action_sampler=d3rlpy.algos.SoftmaxTransformerActionSampler(
            temperature=1.0,
        ),
    logger_adapter=UnifiedFileAdapterFactory(),
)

2025-07-26 18:55.51 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(4,)]), action_signature=Signature(dtype=[dtype('int32')], shape=[(1,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.DISCRETE: 2>, action_size=2)
2025-07-26 18:55.51 [debug    ] Building models...            
2025-07-26 18:55.51 [debug    ] Models have been built.       
2025-07-26 18:55.51 [info     ] Directory is created at d3rlpy_logs/Discrete_DT_Cartpole_20250726185551
2025-07-26 18:55.51 [info     ] Parameters                     params={'observation_shape': [4], 'action_size': 2, 'config': {'type': 'discrete_decision_transformer', 'params': {'batch_size': 128, 'gamma': 0.99, 'observation_scaler': {'type': 'none', 'params': {}}, 'action_scaler': {'type': 'none', 'params': {}}, 'reward_scaler': {'type': 'none', 'params': {}}, 'compile_graph': False, 'context_size': 30, 'max_timeste

Epoch 1/100: 100%|██████████| 1000/1000 [00:35<00:00, 28.54it/s, loss=0.475, learning_rate=0.000599]


2025-07-26 18:56.28 [info     ] New best score                 epoch=1 score=200.0
2025-07-26 18:56.28 [info     ] Saving model 'd3rlpy_logs/Discrete_DT_Cartpole_20250726185551/model_epoch_1.d3' epoch=1
2025-07-26 18:56.28 [info     ] Model parameters are saved to d3rlpy_logs/Discrete_DT_Cartpole_20250726185551/model_epoch_1.d3
2025-07-26 18:56.28 [info     ] Discrete_DT_Cartpole_20250726185551: epoch=1 step=1000 epoch=1 metrics={'time_sample_batch': 0.003898127317428589, 'time_algorithm_update': 0.0309338161945343, 'loss': 0.4749610639810562, 'learning_rate': 0.0005986933474055758, 'time_step': 0.034896363973617554, 'eval_episode_mean_reward': 200.0, 'eval_episode_median_reward': 200.0, 'eval_episode_std_reward': 0.0, 'eval_episode_min_reward': 200.0, 'eval_episode_max_reward': 200.0, 'eval_episode_count': 1.0} step=1000


Epoch 2/100: 100%|██████████| 1000/1000 [00:34<00:00, 28.69it/s, loss=0.457, learning_rate=0.000596]


2025-07-26 18:58.00 [info     ] Discrete_DT_Cartpole_20250726185551: epoch=2 step=2000 epoch=2 metrics={'time_sample_batch': 0.0038970093727111817, 'time_algorithm_update': 0.03074453663825989, 'loss': 0.45654186740517616, 'learning_rate': 0.0005961918837351237, 'time_step': 0.034705580234527585, 'eval_episode_mean_reward': 192.4, 'eval_episode_median_reward': 200.0, 'eval_episode_std_reward': 21.23864402451343, 'eval_episode_min_reward': 102.0, 'eval_episode_max_reward': 200.0, 'eval_episode_count': 50.0} step=2000


...

2025-07-26 19:07.24 [info     ] Discrete_DT_Cartpole_20250726185551: epoch=10 step=10000 epoch=10 metrics={'time_sample_batch': 0.0037932274341583253, 'time_algorithm_update': 0.023890194416046142, 'loss': 0.4282543792128563, 'learning_rate': 0.00046362556350163485, 'time_step': 0.027744596004486085, 'eval_episode_mean_reward': 199.96, 'eval_episode_median_reward': 200.0, 'eval_episode_std_reward': 0.28, 'eval_episode_min_reward': 198.0, 'eval_episode_max_reward': 200.0, 'eval_episode_count': 50.0} step=10000


Epoch 11/100: 100%|██████████| 1000/1000 [00:27<00:00, 35.87it/s, loss=0.422, learning_rate=0.000437]


2025-07-26 19:08.31 [info     ] Discrete_DT_Cartpole_20250726185551: epoch=11 step=11000 epoch=11 metrics={'time_sample_batch': 0.0037911341190338134, 'time_algorithm_update': 0.023924594402313232, 'loss': 0.422409945756197, 'learning_rate': 0.0004365009007334657, 'time_step': 0.027776384353637697, 'eval_episode_mean_reward': 197.94, 'eval_episode_median_reward': 200.0, 'eval_episode_std_reward': 13.859884559403802, 'eval_episode_min_reward': 101.0, 'eval_episode_max_reward': 200.0, 'eval_episode_count': 50.0} step=11000


Epoch 12/100: 100%|██████████| 1000/1000 [00:27<00:00, 35.89it/s, loss=0.416, learning_rate=0.000408]


2025-07-26 19:09.37 [info     ] Discrete_DT_Cartpole_20250726185551: epoch=12 step=12000 epoch=12 metrics={'time_sample_batch': 0.003790334701538086, 'time_algorithm_update': 0.02390558171272278, 'loss': 0.4161187565028667, 'learning_rate': 0.0004078540683044525, 'time_step': 0.027756635904312134, 'eval_episode_mean_reward': 194.6, 'eval_episode_median_reward': 200.0, 'eval_episode_std_reward': 26.570660511172846, 'eval_episode_min_reward': 43.0, 'eval_episode_max_reward': 200.0, 'eval_episode_count': 50.0} step=12000
Early stopping at epoch 12 due to no improvement in the last 10 epochs.
